# Lab 3: File Search Agent — Policy Q&A (RAG)
---
**SmartClaims AI Agent** | Microsoft Foundry Agent Service (v2.x)

## Objective
Build a Retrieval-Augmented Generation (RAG) agent that answers questions using an uploaded insurance policy document. You’ll learn:
- Creating and managing **Vector Stores** for document indexing
- Uploading documents with automatic chunking, embedding, and indexing
- Using **FileSearchTool** to ground agent responses in document content
- Multi-turn policy Q&A with source-grounded answers

## Architecture
```
Policy Document → Upload → Vector Store (chunk → embed → index) → FileSearchTool → Agent → Grounded Answers
```

> 💡 **What is RAG?** Retrieval-Augmented Generation combines document retrieval with LLM generation. The agent searches your documents first, then generates answers grounded in the retrieved content — reducing hallucination and ensuring accuracy.

## Step 1: Import Dependencies

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from utils.config import get_clients, MODEL, POLICY_DOC, print_header, print_step
from azure.ai.projects.models import PromptAgentDefinition, FileSearchTool

project_client, openai_client = get_clients()
print("✅ Clients created")

## Step 2: Create a Vector Store

A **Vector Store** is a managed document index. When you upload documents to it, the service automatically:

1. **Chunks** the document into smaller passages
2. **Embeds** each chunk into a vector representation
3. **Indexes** the vectors for fast similarity search

This is all managed by the platform — no need to configure embedding models or vector databases yourself.

In [ ]:
vector_store = openai_client.vector_stores.create(
    name="ContosoPolicyStore"
)
print(f"Vector Store created — ID: {vector_store.id}")

## Step 3: Upload & Index the Policy Document

`upload_and_poll()` is a convenience method that:
1. Uploads the file to the service
2. Triggers chunking and embedding
3. **Waits** until indexing is complete before returning

This ensures the document is fully searchable before we create the agent.

In [ ]:
file = openai_client.vector_stores.files.upload_and_poll(
    vector_store_id=vector_store.id,
    file=open(str(POLICY_DOC), "rb"), 
    # what does rb mean? - it means "read binary" mode, which is used to read files that are not plain text, such as images or PDFs

    # where is the file located? - the file is located at the path specified by the POLICY_DOC variable, which is defined in the utils.config module
)
print(f"✅ File indexed: {file.id}")

## Step 4: Create the File Search Agent

The **FileSearchTool** connects your agent to the Vector Store. When the agent receives a question, it:
1. Searches the vector store for relevant document passages
2. Uses those passages as context for generating the answer
3. Can cite specific sections from the source document

> 💡 **Agent Instructions Matter:** Notice the instructions tell the agent to cite sections, admit when information isn’t in the document, and be precise with numbers. Good instructions dramatically improve RAG quality.

In [ ]:
file_search_tool = FileSearchTool(
    vector_store_ids=[vector_store.id]
)

agent = project_client.agents.create_version(
    agent_name="smartclaims-policy-qa",
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are SmartClaims Policy Assistant for Contoso Insurance. "
            "Answer questions using ONLY the uploaded policy documents. "
            "Rules: (1) Cite the specific section. (2) If not in the "
            "document, say so. (3) Be precise with numbers and limits. "
            "(4) For exclusions, suggest alternatives if available."
        ),
        tools=[file_search_tool],
    ),
)
print(f"Agent: {agent.name} v{agent.version}")

## Step 5: Run Policy Q&A

Let’s test the agent with a variety of policy questions. We use a conversation to maintain context across questions, simulating a real customer interaction.

In [ ]:
conversation = openai_client.conversations.create()

def ask(q):
    """Send a question to the policy Q&A agent."""
    return openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent_reference": {
            "name": agent.name, "version": agent.version,
            "type": "agent_reference"}},
        input=q,
    )

In [ ]:
questions = [
    "What is the maximum auto liability coverage?",
    "Are flood damages covered under property insurance?",
    "How do I file a claim after an auto accident? What docs do I need?",
    "What happens if someone commits insurance fraud?",
    "What are the health insurance deductible options for HDHP?",
]

for i, q in enumerate(questions, 1):
    print(f"\n{'─'*60}")
    print(f"❓ Question {i}: {q}")
    r = ask(q)
    print(f"🤖 Answer: {r.output_text}")

## Step 6: Inspect Retrieved Chunks (References)

Before cleaning up, let's peek under the hood. The **FileSearchTool** doesn't just return an answer — it retrieves specific **chunks** from the vector store and uses them as context. By inspecting the raw response output, we can see exactly which document passages the agent relied on, including chunk text, relevance scores, and source file details.

> 💡 **Why does this matter?** In production RAG systems, surfacing chunk-level references lets you audit answer quality, debug retrieval issues, and build user-facing citations.

In [ ]:
# Ask a focused question and inspect the full response with chunk details
# Key: include=["file_search_call.results"] tells the API to return chunk details
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {
        "name": agent.name, "version": agent.version,
        "type": "agent_reference"}},
    input="what got affected on 25th january 2024",
    include=["file_search_call.results"],
)

print("=" * 70)
print("🤖 ANSWER")
print("=" * 70)
print(response.output_text)

# Extract and display chunk-level references from the response
print(f"\n{'=' * 70}")
print("📎 RETRIEVED CHUNKS (References)")
print("=" * 70)

chunk_count = 0
for item in response.output:
    # file_search_call items contain the retrieved results
    if item.type == "file_search_call":
        results = item.results or []
        for j, chunk in enumerate(results, 1):
            chunk_count += 1
            print(f"\n--- Chunk {chunk_count} ---")
            print(f"  📄 File ID   : {chunk.file_id}")
            print(f"  📌 Filename  : {getattr(chunk, 'filename', 'N/A')}")
            print(f"  🎯 Score     : {chunk.score:.4f}")
            # Show attributes if available
            for attr in ["attributes", "metadata"]:
                if hasattr(chunk, attr) and getattr(chunk, attr):
                    print(f"  🏷️  {attr.title()}: {getattr(chunk, attr)}")
            text = getattr(chunk, "text", None)
            if text:
                preview = text[:500] + ("..." if len(text) > 500 else "")
                print(f"  📝 Text      :\n{preview}")

if chunk_count == 0:
    print("  (No chunk details exposed — try enabling include_search_results)")
else:
    print(f"\n✅ Total chunks retrieved: {chunk_count}")

In [ ]:
# Ask a focused question and inspect the full response with chunk details
# Key: include=["file_search_call.results"] tells the API to return chunk details
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {
        "name": agent.name, "version": agent.version,
        "type": "agent_reference"}},
    input="what got affected on 25th january 2024",
    include=["file_search_call.results"],
)

print("=" * 70)
print("🤖 ANSWER")
print("=" * 70)
print(response.output_text)

# Extract and display chunk-level references from the response
print(f"\n{'=' * 70}")
print("📎 RETRIEVED CHUNKS (References)")
print("=" * 70)

chunk_count = 0
for item in response.output:
    # file_search_call items contain the retrieved results
    if item.type == "file_search_call":
        results = item.results or []
        for j, chunk in enumerate(results, 1):
            chunk_count += 1
            print(f"\n--- Chunk {chunk_count} ---")
            print(f"  📄 File ID   : {chunk.file_id}")
            print(f"  📌 Filename  : {getattr(chunk, 'filename', 'N/A')}")
            print(f"  🎯 Score     : {chunk.score:.4f}")
            # Show attributes if available
            for attr in ["attributes", "metadata"]:
                if hasattr(chunk, attr) and getattr(chunk, attr):
                    print(f"  🏷️  {attr.title()}: {getattr(chunk, attr)}")
            text = getattr(chunk, "text", None)
            if text:
                preview = text[:500] + ("..." if len(text) > 500 else "")
                print(f"  📝 Text      :\n{preview}")

if chunk_count == 0:
    print("  (No chunk details exposed — try enabling include_search_results)")
else:
    print(f"\n✅ Total chunks retrieved: {chunk_count}")

## Step 7: Clean Up
Delete the agent and vector store to free resources.

In [ ]:
project_client.agents.delete(agent.name)
openai_client.vector_stores.delete(vector_store.id)
print("✅ Agent and Vector Store deleted")

## ✅ Lab 3 Complete!

### Key Takeaways
| Concept | What You Learned |
|---------|-----------------|
| Vector Store | Managed document index with auto chunk/embed/index |
| FileSearchTool | Connects agent to vector store for RAG |
| RAG Pattern | Retrieve → Augment → Generate for grounded answers |
| Citation | Agent can reference specific document sections |
| Conversation | Multi-turn Q&A with context preservation |

### RAG Best Practices
- Write clear instructions telling the agent to cite sources
- Tell the agent what to do when information isn’t found
- Use specific, well-structured documents for better retrieval
- Test with questions that have definitive answers in the document

---
**Next →** [Lab 4: Code Interpreter](lab4_code_interpreter.ipynb) — Analyze claims data with AI-generated Python code